# TAC-LAnoBERT v2 Optimized (KNN + PCA)

**Purpose**: Run optimized inference with improvements from code changes:

## 🚀 Current Optimizations

### Phase 1: KNN + PCA (Current)
- ✅ **KNN Distance** (not Mahalanobis - failed with AUROC 0.37)
- ✅ **PCA 768→64 dims** (addresses distance concentration)
- ✅ **Alpha = 0.85** (MLM dominant, KNN as FP suppressor)
- ✅ **Queue = 1024** (larger for better KNN coverage)
- ✅ **Ring Buffer** (zero-allocation memory queue)
- ✅ **Binary Search DLT** (optimized early detection metrics)

### Phase 2: Projection Head (Next)
- 🔄 **768→256→64 projection** (dedicated early detection space)
- 🔄 **Separate training phase** (after Phase 1 validation)

---

## 📊 Expected Results

| Metric | Baseline (v2 2-epoch) | Target (Optimized) | Status |
|--------|----------------------|-------------------|--------|
| F1     | 0.912                | ≥ 0.985           | 🎯     |
| FP     | 4,806                | ≤ 1,000           | 🎯     |
| EWR    | 32.94%               | ≥ 30%             | 🎯     |
| AUROC  | 0.980                | ≥ 0.99            | 🎯     |

---

## ⚙️ Configuration

**Config**: `configs/bgl_tac_v2_optimized.yaml`

**Reuses**:
- ✅ Pre-trained model (10 epochs from `BGL_tac_v2_2epochs`)
- ✅ Existing tokenizer
- ✅ Preprocessed BGL data

**New Results**: `outputs/BGL_tac_v2_optimized/results/`

---

## 🕐 Runtime

- **GPU (T4)**: ~45-60 min (inference only, no training)
- **CPU**: ~2-3 hours

---

## 1. Setup Environment

In [ ]:
# Clone repository (if on Kaggle/Colab)
import os

if not os.path.exists('TAC-LAnoBERT-y'):
    print("📦 Cloning repository...")
    !git clone https://github.com/rubyhcm/TAC-LAnoBERT-y.git
    %cd TAC-LAnoBERT-y
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")
    if not os.getcwd().endswith('TAC-LAnoBERT-y'):
        %cd TAC-LAnoBERT-y
        print(f"📂 Changed to: {os.getcwd()}")

In [ ]:
# Install dependencies
print("📦 Installing dependencies...")
!pip install -r requirements.txt -q
print("✅ Dependencies installed")

In [ ]:
# Verify environment
import torch
import transformers
import numpy as np
import sys
from pathlib import Path

print("=" * 70)
print("ENVIRONMENT INFO")
print("=" * 70)
print(f"\nPython:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"\nCUDA:         {'✅ Available' if torch.cuda.is_available() else '❌ Not available (CPU mode)'}")
if torch.cuda.is_available():
    print(f"  GPU:        {torch.cuda.get_device_name(0)}")
    print(f"  Version:    {torch.version.cuda}")
    print(f"  Memory:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("  ⚠️  Running on CPU (inference will be slower)")

print(f"\nWorking dir:  {os.getcwd()}")
print("\n" + "=" * 70)
print("✅ Environment ready")
print("=" * 70)

## 2. Check Prerequisites

Verify all required data and models are available.

In [ ]:
# Check and copy datasets from Kaggle input
import glob
import shutil

print("=" * 70)
print("CHECKING PREREQUISITES")
print("=" * 70)

def copy_from_kaggle_input(pattern, target_dir, name):
    """Search for and copy data from Kaggle input"""
    found = glob.glob(pattern, recursive=True)
    if found:
        src = found[0]
        if os.path.isdir(src):
            dst = os.path.join(target_dir, os.path.basename(src))
            if not os.path.exists(dst):
                print(f"\n📦 {name}: Found in input, copying...")
                print(f"   {src} → {dst}")
                os.makedirs(target_dir, exist_ok=True)
                shutil.copytree(src, dst)
                print("   ✅ Copied")
            else:
                print(f"\n✅ {name}: Already in working directory")
        else:
            # Single file
            os.makedirs(target_dir, exist_ok=True)
            for f in glob.glob(os.path.join(os.path.dirname(src), "*")):
                dst_file = os.path.join(target_dir, os.path.basename(f))
                if not os.path.exists(dst_file):
                    shutil.copy2(f, dst_file)
            print(f"\n✅ {name}: Copied from input")
        return True
    return False

# 1. TAC v2 trained model (REQUIRED - 10 epochs)
if not copy_from_kaggle_input(
    "/kaggle/input/**/BGL_tac_v2_2epochs",
    "outputs",
    "TAC v2 Model (10 epochs)"
):
    if os.path.exists("outputs/BGL_tac_v2_2epochs/model"):
        print("\n✅ TAC v2 Model: Already available")
    else:
        print("\n❌ TAC v2 Model: NOT FOUND!")
        print("   → Attach 'BGL_tac_v2_2epochs' dataset as Kaggle input")
        print("   → This is the 10-epoch trained model")

# 2. BGL preprocessed data (REQUIRED)
if not copy_from_kaggle_input(
    "/kaggle/input/**/BGL/BGL_test_parsed.log",
    "data",
    "BGL Data"
):
    if os.path.exists("data/BGL/BGL_test_parsed.log"):
        print("\n✅ BGL Data: Already available")
    else:
        print("\n❌ BGL Data: NOT FOUND!")
        print("   → Attach BGL preprocessed dataset as Kaggle input")

# 3. Baseline results (OPTIONAL - for comparison)
baseline_found = False
for baseline_name, baseline_path in [
    ("LAnoBERT", "/kaggle/input/**/BGL_lanobert"),
    ("TAC Original", "/kaggle/input/**/BGL_tac")
]:
    if copy_from_kaggle_input(baseline_path, "outputs", f"Baseline ({baseline_name})"):
        baseline_found = True

if not baseline_found:
    print("\n⚠️  No baseline results found (will skip comparison)")
    print("   Optional: Attach baseline datasets for comparison")

print("\n" + "=" * 70)

In [ ]:
# Verify critical files
print("=" * 70)
print("FILE VERIFICATION")
print("=" * 70)

critical_files = {
    "Model Config": "outputs/BGL_tac_v2_2epochs/model/config.json",
    "Model Weights": "outputs/BGL_tac_v2_2epochs/model/model.safetensors",
    "Time2Vec Weights": "outputs/BGL_tac_v2_2epochs/model/time2vec.pt",
    "Tokenizer": "outputs/BGL_tac_v2_2epochs/tokenizer/BGL_LogBERT-vocab.txt",
    "Test Data": "data/BGL/BGL_test_parsed.log",
    "Test Labels": "data/BGL/BGL_test_label.log",
    "Test Timestamps": "data/BGL/BGL_test_parsed.timestamps",
}

all_ok = True
print("\nRequired files:")
for name, path in critical_files.items():
    # Check alternative paths (e.g., model/final/)
    if not os.path.exists(path) and "model/" in path:
        alt_path = path.replace("model/", "model/final/")
        if os.path.exists(alt_path):
            path = alt_path
    
    if os.path.exists(path):
        size = os.path.getsize(path)
        size_mb = size / (1024 * 1024)
        print(f"  ✅ {name:<20} ({size_mb:>8.2f} MB)")
    else:
        print(f"  ❌ {name:<20} NOT FOUND")
        print(f"     Expected: {path}")
        all_ok = False

print("\n" + "=" * 70)
if all_ok:
    print("✅ ALL FILES READY")
    print("=" * 70)
else:
    print("❌ MISSING FILES - Cannot proceed")
    print("=" * 70)
    raise FileNotFoundError("Required files missing. Check Kaggle input datasets.")

## 3. View Configuration

Review the optimized configuration settings.

In [ ]:
# Load and display configuration
import yaml

config_path = "configs/bgl_tac_v2_optimized.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

print("=" * 70)
print("OPTIMIZED CONFIGURATION")
print("=" * 70)

print(f"\n📋 Basic Info:")
print(f"   Dataset:     {config['dataset']}")
print(f"   Run name:    {config['run_name']}")
print(f"   Config:      {config_path}")

# TAC settings
tac = config.get('tac', {})
print(f"\n⏰ TAC Features:")
print(f"   Enabled:     {tac.get('enabled', False)}")
print(f"   Mode:        {tac.get('mode', 'N/A')}")
print(f"   Time2Vec:    {tac.get('time2vec', {}).get('enabled', False)}")

# Memory Queue (KNN)
memory = tac.get('memory', {})
print(f"\n🧠 Memory Queue (KNN):")
print(f"   Enabled:       {memory.get('enabled', False)}")
print(f"   Capacity:      {memory.get('queue_capacity', 'N/A')}")
print(f"   Distance:      {memory.get('distance_metric', 'N/A')}")
print(f"   K-neighbors:   {memory.get('k_neighbors', 'N/A')}")
print(f"   PCA dims:      {memory.get('pca_components', 'N/A')} (768→{memory.get('pca_components', 'N/A')})")

# Hybrid Scoring
scoring = tac.get('scoring', {})
print(f"\n📊 Hybrid Scoring:")
print(f"   Alpha:         {scoring.get('alpha', 'N/A')} (MLM weight)")
print(f"   MLM:           {scoring.get('alpha', 0.5)*100:.0f}%")
print(f"   KNN:           {(1-scoring.get('alpha', 0.5))*100:.0f}%")
print(f"   Normalize:     {scoring.get('normalize_scores', False)}")

# v2 Improvements
if 'tac_v2' in config:
    v2 = config['tac_v2']
    print(f"\n🚀 TAC v2 Improvements:")
    print(f"   Enabled:             {v2.get('enabled', False)}")
    print(f"   Early detection:     {v2.get('early_detection_loss', {}).get('enabled', False)}")
    print(f"   Temporal features:   {v2.get('temporal_features', {}).get('enabled', False)}")
    print(f"   Data augmentation:   {v2.get('data_augmentation', {}).get('enabled', False)}")
    print(f"   Curriculum learning: {v2.get('curriculum_learning', {}).get('enabled', False)}")
    print(f"   Improved scoring:    {v2.get('improved_scoring', {}).get('enabled', False)}")
    
    # Projection head (Phase 2)
    proj = v2.get('projection_head', {})
    print(f"\n🎯 Projection Head (Phase 2):")
    print(f"   Enabled:       {proj.get('enabled', False)}")
    if proj.get('enabled'):
        print(f"   Architecture:  {proj.get('input_dim')}→{proj.get('hidden_dim')}→{proj.get('output_dim')}")
        print(f"   Checkpoint:    {proj.get('checkpoint', 'Not trained yet')}")
    else:
        print(f"   Status:        Not enabled (use Phase 1 first)")

# Evaluation settings
eval_cfg = v2.get('evaluation', {})
if eval_cfg:
    print(f"\n📈 Evaluation:")
    print(f"   Standard metrics:  {eval_cfg.get('compute_standard', True)}")
    print(f"   DLT/EWR:          {eval_cfg.get('compute_dlt', False)}")
    if eval_cfg.get('compute_dlt'):
        print(f"   DLT intervals:    {eval_cfg.get('dlt_intervals', [])}")
    print(f"   ROI analysis:     {eval_cfg.get('compute_roi', False)}")
    print(f"   Alert fatigue:    {eval_cfg.get('compute_alert_fatigue', False)}")

print("\n" + "=" * 70)

## 4. Run Optimized Inference 🚀

Run inference with optimized settings:
- KNN distance (k=10)
- PCA 768→64 dims
- Alpha = 0.85 (MLM dominant)
- Queue = 1024

**No training needed** - reuses 10-epoch model from Phase 2.

In [ ]:
# Check if inference already done
results_dir = Path("outputs/BGL_tac_v2_optimized/results")
score_files = list(results_dir.glob("scores_*.npy")) if results_dir.exists() else []

if score_files:
    print("✅ Inference results already exist")
    print(f"   Found {len(score_files)} score file(s) in {results_dir}")
    print("\n   Score files:")
    for sf in sorted(score_files):
        print(f"     • {sf.name}")
    print("\n   Skipping inference (remove results/ to re-run)")
    skip_inference = True
else:
    print("📊 No existing results found")
    print(f"   Will run inference and save to: {results_dir}")
    skip_inference = False

In [ ]:
# Run inference
import time

if not skip_inference:
    print("=" * 70)
    print("STARTING OPTIMIZED INFERENCE")
    print("=" * 70)
    print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("\nThis will take ~45-60 min on GPU (T4)...\n")
    
    start_time = time.time()
    
    # Run TAC v2 inference with optimized config
    !python -m tac_lanobert.inference_tac --config configs/bgl_tac_v2_optimized.yaml
    
    end_time = time.time()
    duration = end_time - start_time
    minutes = int(duration // 60)
    seconds = int(duration % 60)
    
    print("\n" + "=" * 70)
    print("INFERENCE COMPLETE")
    print("=" * 70)
    print(f"End time:  {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Duration:  {minutes}m {seconds}s")
    print(f"Results:   {results_dir}")
    print("=" * 70)
else:
    print("\nℹ️  Using existing results (inference skipped)")

## 5. View Results

Load and display detailed metrics from the optimized run.

In [ ]:
# Load score files
results_dir = Path("outputs/BGL_tac_v2_optimized/results")
score_files = list(results_dir.glob("scores_*.npy"))

print("=" * 70)
print("SCORE STATISTICS")
print("=" * 70)

if score_files:
    print(f"\nFound {len(score_files)} score file(s):\n")
    
    scores_dict = {}
    for sf in sorted(score_files):
        scores = np.load(sf)
        name = sf.stem.replace('scores_', '')
        scores_dict[name] = scores
        
        print(f"{name}:")
        print(f"  Lines:       {len(scores):,}")
        print(f"  Range:       [{scores.min():.6f}, {scores.max():.6f}]")
        print(f"  Mean ± Std:  {scores.mean():.6f} ± {scores.std():.6f}")
        print(f"  Median:      {np.median(scores):.6f}")
        print(f"  Quartiles:   Q1={np.percentile(scores, 25):.6f}, Q3={np.percentile(scores, 75):.6f}")
        print()
else:
    print("\n❌ No score files found!")
    print(f"   Expected in: {results_dir}")
    print("   Check if inference completed successfully.")

print("=" * 70)

In [ ]:
# Parse metrics from report file
import re
import json

def parse_report(report_path):
    """Parse TAC report file for metrics"""
    if not os.path.exists(report_path):
        return None
    
    with open(report_path, 'r') as f:
        content = f.read()
    
    metrics = {}
    
    # Basic metrics
    patterns = {
        'auroc': r'AUROC:\s+([0-9.e+-]+)',
        'f1': r'best_F1:\s+([0-9.e+-]+)',
        'precision': r'best_precision:\s+([0-9.e+-]+)',
        'recall': r'best_recall:\s+([0-9.e+-]+)',
        'threshold': r'best_threshold:\s+([0-9.e+-]+)',
    }
    
    for key, pattern in patterns.items():
        if m := re.search(pattern, content):
            metrics[key] = float(m.group(1))
    
    # Confusion matrix
    cm_pattern = r'confusion_matrix:.*?\[\[\s*(\d+)\s+(\d+)\s*\]\s*\[\s*(\d+)\s+(\d+)\s*\]\]'
    if m := re.search(cm_pattern, content, re.DOTALL):
        tn, fp, fn, tp = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        metrics.update({
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
            'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        })
    
    # Early detection metrics (if available)
    if m := re.search(r'DLT_mean:\s+([0-9.e+-]+)', content):
        metrics['dlt_mean'] = float(m.group(1))
    if m := re.search(r'EWR:\s+([0-9.e+-]+)', content):
        metrics['ewr'] = float(m.group(1))
    
    return metrics if metrics else None

# Find and parse optimized report
report_files = list(results_dir.glob("*_report.txt"))

print("=" * 70)
print("OPTIMIZED RESULTS")
print("=" * 70)

opt_metrics = None
if report_files:
    opt_metrics = parse_report(str(report_files[0]))
    
    if opt_metrics:
        print(f"\n📊 Performance Metrics:\n")
        print(f"  F1-Score:    {opt_metrics.get('f1', 0):.6f}")
        print(f"  Precision:   {opt_metrics.get('precision', 0):.6f}")
        print(f"  Recall:      {opt_metrics.get('recall', 0):.6f}")
        print(f"  AUROC:       {opt_metrics.get('auroc', 0):.6f}")
        print(f"  FPR:         {opt_metrics.get('fpr', 0)*100:.4f}%")
        
        print(f"\n🎯 Detection Metrics:\n")
        print(f"  Threshold:   {opt_metrics.get('threshold', 0):.6f}")
        print(f"  True Pos:    {opt_metrics.get('tp', 0):>8,}")
        print(f"  False Pos:   {opt_metrics.get('fp', 0):>8,}")
        print(f"  True Neg:    {opt_metrics.get('tn', 0):>8,}")
        print(f"  False Neg:   {opt_metrics.get('fn', 0):>8,}")
        
        if 'dlt_mean' in opt_metrics:
            print(f"\n⏰ Early Detection:\n")
            print(f"  DLT (mean):  {opt_metrics['dlt_mean']:.1f} seconds")
            if 'ewr' in opt_metrics:
                print(f"  EWR:         {opt_metrics['ewr']*100:.2f}%")
        
        # Check targets
        print(f"\n✅ Target Achievement:\n")
        targets = [
            ('F1 ≥ 0.985', opt_metrics.get('f1', 0) >= 0.985),
            ('FP ≤ 1,000', opt_metrics.get('fp', 999999) <= 1000),
            ('EWR ≥ 30%', opt_metrics.get('ewr', 0) >= 0.30),
            ('AUROC ≥ 0.99', opt_metrics.get('auroc', 0) >= 0.99),
        ]
        for target_name, achieved in targets:
            icon = "✅" if achieved else "❌"
            print(f"  {icon} {target_name}")
    else:
        print("\n⚠️  Could not parse report file")
else:
    print("\n⚠️  No report files found")
    print(f"   Expected: *_report.txt in {results_dir}")

print("\n" + "=" * 70)

## 6. Comparison with Baseline

Compare optimized results with previous versions.

In [ ]:
# Find all available baseline results
baselines = {}

# TAC v2 2-epoch (original)
tac_v2_reports = list(Path("outputs/BGL_tac_v2_2epochs/results").glob("*_report.txt"))
if tac_v2_reports:
    metrics = parse_report(str(tac_v2_reports[0]))
    if metrics:
        baselines['TAC v2 (2-epoch)'] = metrics

# TAC KNN variant
tac_knn_reports = list(Path("outputs/BGL_tac_knn/results").glob("*_report.txt")) if Path("outputs/BGL_tac_knn").exists() else []
if tac_knn_reports:
    metrics = parse_report(str(tac_knn_reports[0]))
    if metrics:
        baselines['TAC KNN (alpha=0.5)'] = metrics

# Original TAC
if Path("outputs/BGL_tac/results/BGL_tac_hybrid_report.txt").exists():
    metrics = parse_report("outputs/BGL_tac/results/BGL_tac_hybrid_report.txt")
    if metrics:
        baselines['TAC (original)'] = metrics

# LAnoBERT baseline
if Path("outputs/BGL_lanobert/results/BGL_error_mean_report.txt").exists():
    metrics = parse_report("outputs/BGL_lanobert/results/BGL_error_mean_report.txt")
    if metrics:
        baselines['LAnoBERT (baseline)'] = metrics

print(f"Found {len(baselines)} baseline(s) for comparison:")
for name in baselines.keys():
    print(f"  • {name}")

In [ ]:
# Compare with each baseline
if baselines and opt_metrics:
    for baseline_name, baseline_metrics in baselines.items():
        print("\n" + "=" * 70)
        print(f"COMPARISON: {baseline_name} → Optimized")
        print("=" * 70)
        
        # Metrics to compare
        comparisons = [
            ('F1-Score', 'f1', 'higher'),
            ('Precision', 'precision', 'higher'),
            ('Recall', 'recall', 'higher'),
            ('AUROC', 'auroc', 'higher'),
            ('FPR', 'fpr', 'lower'),
        ]
        
        print(f"\n{'Metric':<12} {'Baseline':<12} {'Optimized':<12} {'Δ':<12} {'Status'}")
        print("-" * 70)
        
        for metric_name, metric_key, better_direction in comparisons:
            base_val = baseline_metrics.get(metric_key)
            opt_val = opt_metrics.get(metric_key)
            
            if base_val is not None and opt_val is not None:
                delta = opt_val - base_val
                delta_pct = (delta / base_val * 100) if base_val != 0 else 0
                
                # Determine status
                if better_direction == 'lower':
                    status = "✅ Better" if delta < 0 else ("❌ Worse" if delta > 0 else "≈ Same")
                else:
                    status = "✅ Better" if delta > 0 else ("❌ Worse" if delta < 0 else "≈ Same")
                
                # Format values
                if metric_key == 'fpr':
                    base_str = f"{base_val*100:.4f}%"
                    opt_str = f"{opt_val*100:.4f}%"
                    delta_str = f"{delta_pct:+.2f}%"
                else:
                    base_str = f"{base_val:.6f}"
                    opt_str = f"{opt_val:.6f}"
                    delta_str = f"{delta_pct:+.2f}%"
                
                print(f"{metric_name:<12} {base_str:<12} {opt_str:<12} {delta_str:<12} {status}")
            else:
                print(f"{metric_name:<12} {'N/A':<12} {'N/A':<12} {'N/A':<12} {'N/A'}")
        
        # Alert volume comparison
        if all(k in baseline_metrics for k in ['tp', 'fp']) and all(k in opt_metrics for k in ['tp', 'fp']):
            base_alerts = baseline_metrics['tp'] + baseline_metrics['fp']
            opt_alerts = opt_metrics['tp'] + opt_metrics['fp']
            alert_reduction = (base_alerts - opt_alerts) / base_alerts * 100 if base_alerts > 0 else 0
            
            print(f"\n🔔 Alert Volume:")
            print(f"   Baseline:  {base_alerts:>8,} alerts")
            print(f"   Optimized: {opt_alerts:>8,} alerts")
            print(f"   Reduction: {alert_reduction:>8.2f}%")
            
            if alert_reduction > 0:
                print(f"\n   ✅ {alert_reduction:.1f}% fewer alerts (reduced alert fatigue!)")
        
        print("\n" + "=" * 70)
        
elif not baselines:
    print("\n⚠️  No baselines found for comparison")
elif not opt_metrics:
    print("\n⚠️  Optimized metrics not available")

## 7. Summary & Next Steps

In [ ]:
from datetime import datetime

print("=" * 70)
print("OPTIMIZATION SUMMARY")
print("=" * 70)

print(f"\n🎯 Configuration: {config['run_name']}")
print(f"   Config file:   configs/bgl_tac_v2_optimized.yaml")
print(f"   Results dir:   {results_dir}")
print(f"   Completed:     {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if opt_metrics:
    print(f"\n📊 Key Results:")
    print(f"   F1:        {opt_metrics.get('f1', 0):.6f}")
    print(f"   Precision: {opt_metrics.get('precision', 0):.6f}")
    print(f"   Recall:    {opt_metrics.get('recall', 0):.6f}")
    print(f"   AUROC:     {opt_metrics.get('auroc', 0):.6f}")
    print(f"   FPR:       {opt_metrics.get('fpr', 0)*100:.4f}%")
    print(f"   FP Count:  {opt_metrics.get('fp', 0):,}")
    
    if 'ewr' in opt_metrics:
        print(f"   EWR:       {opt_metrics['ewr']*100:.2f}%")
    if 'dlt_mean' in opt_metrics:
        print(f"   DLT:       {opt_metrics['dlt_mean']:.1f}s")

# Show improvements vs best baseline
if baselines and opt_metrics:
    # Find baseline with closest/comparable setup
    ref_baseline = baselines.get('TAC v2 (2-epoch)') or baselines.get('TAC KNN (alpha=0.5)')
    if ref_baseline:
        print(f"\n🚀 Improvements vs Reference Baseline:")
        for metric_name, metric_key in [('F1', 'f1'), ('FP', 'fp'), ('AUROC', 'auroc')]:
            if metric_key in ref_baseline and metric_key in opt_metrics:
                base_val = ref_baseline[metric_key]
                opt_val = opt_metrics[metric_key]
                if metric_key == 'fp':
                    delta = opt_val - base_val
                    print(f"   {metric_name}: {base_val:,} → {opt_val:,} ({delta:+,})")
                else:
                    pct_change = (opt_val - base_val) / base_val * 100 if base_val != 0 else 0
                    print(f"   {metric_name}: {base_val:.6f} → {opt_val:.6f} ({pct_change:+.2f}%)")

print("\n" + "=" * 70)
print("NEXT STEPS")
print("=" * 70)

if opt_metrics:
    # Assess if targets are met
    f1_ok = opt_metrics.get('f1', 0) >= 0.985
    fp_ok = opt_metrics.get('fp', 999999) <= 1000
    ewr_ok = opt_metrics.get('ewr', 0) >= 0.30
    auroc_ok = opt_metrics.get('auroc', 0) >= 0.99
    
    targets_met = sum([f1_ok, fp_ok, ewr_ok, auroc_ok])
    
    if targets_met >= 3:
        print("\n✅ Phase 1 targets achieved ({targets_met}/4)!")
        print("\n📋 Recommended actions:")
        print("   1. ✅ Document Phase 1 results")
        print("   2. 🔄 Proceed to Phase 2: Projection Head training")
        print("   3. 📊 Run ablation studies (alpha, k, PCA dims)")
        print("   4. 🚀 Prepare for deployment if all targets met")
    else:
        print(f"\n⚠️  Phase 1 targets partially met ({targets_met}/4)")
        print("\n📋 Recommended actions:")
        print("   1. 🔍 Analyze failure cases (high FP/FN)")
        print("   2. ⚙️  Tune hyperparameters (alpha, k, PCA dims)")
        print("   3. 📊 Review score distributions")
        print("   4. 🔄 Consider alternative distance metrics")
else:
    print("\n⚠️  No metrics available - check inference logs")

print("\n" + "=" * 70)
print("✅ NOTEBOOK COMPLETE")
print("=" * 70)

## 8. Export Results (Optional)

Save summary for comparison or documentation.

In [ ]:
# Export summary to JSON
if opt_metrics:
    summary = {
        'config': config['run_name'],
        'timestamp': datetime.now().isoformat(),
        'metrics': opt_metrics,
        'configuration': {
            'alpha': scoring.get('alpha'),
            'k_neighbors': memory.get('k_neighbors'),
            'pca_components': memory.get('pca_components'),
            'queue_capacity': memory.get('queue_capacity'),
            'distance_metric': memory.get('distance_metric'),
        },
        'baselines': baselines,
    }
    
    output_json = results_dir / "optimization_summary.json"
    with open(output_json, 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    
    print(f"✅ Summary exported to: {output_json}")
    print(f"\n   Use this for comparison with future experiments.")
else:
    print("⚠️  No metrics to export")